In [1]:
"""Sage-Decoder notebook import bootstrap."""
from pathlib import Path
import sys

def _find_repo_root(start):
    for candidate in (start, *start.parents):
        if (candidate / "isomorphism").is_dir() and (candidate / "pyproject.toml").is_file():
            return candidate
    raise RuntimeError("Could not find the Sage-Decoder repository root.")

repo_root = _find_repo_root(Path.cwd())
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))


# 4.8.8 color code

This notebook reproduces the 4.8.8 color-code example from `Decoupling_topological_CSS_codes.pdf`. Unlike the two-polynomial examples, this case starts from the full `epsilon` matrix, so the notebook calls the general excitation-map API.

The cells are grouped by purpose: input construction, representation matrices, period tables, and decoupling. Computation cells are separated from display cells so any diagnostic text remains isolated from the values that should be checked.

In [2]:
from sage.all import Matrix, block_matrix, zero_matrix

from isomorphism import (
    R,
    build_quotient_translation_action,
    choose_smallest_oblique_cell,
    construct_excitation_map,
    decouple_coarse_matrix,
    oblique_coarse_grain,
    periods_from_generators,
    periods_from_translation_action,
    x,
    y,
)
from isomorphism.css import check_commutation

The matrix below is the `H_X` block from the paper. The full CSS excitation map places this block in the top `H_X` sector and the matching `H_Z` block in the bottom sector. The commutation assertion checks that this full matrix is a valid CSS input before decoupling.

In [3]:
check_matrix_488 = Matrix(
    R,
    [
        [1 + y, 1 + y, 1 + x, 1 + x],
        [x * y, y, x * y, x],
    ],
)
zero_2x4 = zero_matrix(R, 2, 4)
epsilon_488 = block_matrix(
    R,
    2,
    2,
    [[check_matrix_488, zero_2x4], [zero_2x4, check_matrix_488]],
)

assert check_commutation(epsilon_488, num_qubits=4)

In [4]:
epsilon_488

[y + 1 y + 1 x + 1 x + 1|    0     0     0     0]
[  x*y     y   x*y     x|    0     0     0     0]
[-----------------------+-----------------------]
[    0     0     0     0|y + 1 y + 1 x + 1 x + 1]
[    0     0     0     0|  x*y     y   x*y     x]

This block constructs the finite translation representation for the check matrix alone. The expected actions of `x` and `y` are both the two-dimensional unipotent matrix shown in Eq. (VI.7), and their finite order gives period `L = 2`.

In [5]:
representation_488 = build_quotient_translation_action(check_matrix_488, diagnostics=True)

In [6]:
expected_action_488 = Matrix([[1, 0], [1, 1]])
assert representation_488.tx == expected_action_488
assert representation_488.ty == expected_action_488

representation_488.tx, representation_488.ty, representation_488.diagnostics

(
[1 0]  [1 0]                                                                                                     
[1 1], [1 1], {'generator_count': 8, 'standard_basis_size': 8, 'quotient_dimension': 2, 'monomial_basis_size': 2}
)

In [7]:
periods_488 = periods_from_translation_action(
    representation_488.tx,
    representation_488.ty,
    max_period=5,
)
assert periods_488.square_period == 2
assert (2, 0) in periods_488.vectors
assert (1, 1) in periods_488.vectors

periods_488.vectors, periods_488.square_period

(((0, 2), (1, 1), (2, 0), (2, 2)), 2)

The next block repeats the representation construction for the full `epsilon` matrix. This is a direct test of the general API path used when the excitation map is not determined by a single pair `(f, g)`.

In [8]:
full_representation_488 = build_quotient_translation_action(epsilon_488, diagnostics=True)
full_periods_488 = periods_from_translation_action(
    full_representation_488.tx,
    full_representation_488.ty,
    max_period=5,
)

In [9]:
assert full_periods_488.square_period == 2
(
    full_representation_488.tx,
    full_representation_488.ty,
    full_periods_488.vectors,
    full_representation_488.diagnostics,
)

(
[1 0 0 0]  [1 0 0 0]                                                                                                                                         
[1 1 0 0]  [1 1 0 0]                                                                                                                                         
[0 0 1 0]  [0 0 1 0]                                                                                                                                         
[0 0 1 1], [0 0 1 1], ((0, 2), (1, 1), (2, 0), (2, 2)), {'generator_count': 16, 'standard_basis_size': 16, 'quotient_dimension': 4, 'monomial_basis_size': 4}
)

This block decouples the full excitation map after oblique coarse graining. The full-matrix period data computed above chooses the coarse cell explicitly so the decoupling step does not need to rebuild the translation action.

In [10]:
cell_488 = choose_smallest_oblique_cell(full_periods_488.vectors)
coarse_epsilon_488 = oblique_coarse_grain(epsilon_488, *cell_488)
result_488 = decouple_coarse_matrix(
    coarse_epsilon_488,
    num_x_checks=coarse_epsilon_488.nrows() // 2,
    num_qubits=coarse_epsilon_488.ncols() // 2,
)

In [11]:
(
    (result_488.diagnostics["product_x_rank"], result_488.diagnostics["product_z_rank"]),
    cell_488,
    result_488.inverse_maps.phi1_inverse.nrows(),
    result_488.inverse_maps.phi1_inverse.ncols(),
)

((2, 2), ((0, 2), (1, 1)), 8, 8)

The last block optionally fixes the paper's displayed oblique cell `((2, 0), (1, 1))`. This gives a second decoupling check for the same 4.8.8 input using a user-selected coarse-graining cell.

In [12]:
paper_cell_488 = ((2, 0), (1, 1))
coarse_epsilon_488 = oblique_coarse_grain(epsilon_488, *paper_cell_488)
paper_cell_result_488 = decouple_coarse_matrix(
    coarse_epsilon_488,
    num_x_checks=4,
    num_qubits=8,
)

In [13]:
(
    (paper_cell_result_488.diagnostics["product_x_rank"], paper_cell_result_488.diagnostics["product_z_rank"]),
    paper_cell_result_488.inverse_maps.phi1_inverse.nrows(),
)

((2, 2), 8)

In [14]:
(paper_cell_result_488.inverse_maps.phi0_inverse.inverse()) * (coarse_epsilon_488[:4,:8]) * (paper_cell_result_488.inverse_maps.phi1_inverse)

[    1     0     0     0     0     0     0     0]
[    0     1     0     0     0     0     0     0]
[    0     0     0     0 x + 1 y + 1     0     0]
[    0     0     0     0     0     0 x + 1 y + 1]

In [15]:
# Supplement QCA check: diag(phi1_inverse, phi1_dagger).
from isomorphism.chain_maps.decoupling import (
    qca_decoupled_excitation_matrix,
    qca_symplectic_matrix,
    stabilizer_redefined_excitation_matrix,
    target_excitation_matrix,
    verify_qca_decoupling,
)

smallest_cell_coarse_epsilon_488 = oblique_coarse_grain(epsilon_488, *cell_488)
paper_cell_coarse_epsilon_488 = oblique_coarse_grain(epsilon_488, *paper_cell_488)

qca_checks_488 = []
for label, epsilon_input, decoupled in (
    ("smallest_cell", smallest_cell_coarse_epsilon_488, result_488),
    ("paper_cell", paper_cell_coarse_epsilon_488, paper_cell_result_488),
):
    qca_matrix = qca_symplectic_matrix(decoupled.maps, decoupled.inverse_maps)
    qca_product = qca_decoupled_excitation_matrix(
        epsilon_input, decoupled.maps, decoupled.inverse_maps
    )
    target_epsilon = target_excitation_matrix(decoupled)
    redefined_product = stabilizer_redefined_excitation_matrix(decoupled)
    assert redefined_product == target_epsilon
    assert verify_qca_decoupling(decoupled)
    qca_checks_488.append(
        {
            "label": label,
            "qca_matrix_shape": (qca_matrix.nrows(), qca_matrix.ncols()),
            "raw_qca_product_matches_target": qca_product == target_epsilon,
            "after_stabilizer_redefinition_matches_target": redefined_product == target_epsilon,
        }
    )

qca_checks_488


[{'label': 'smallest_cell',
  'qca_matrix_shape': (16, 16),
  'raw_qca_product_matches_target': False,
  'after_stabilizer_redefinition_matches_target': True},
 {'label': 'paper_cell',
  'qca_matrix_shape': (16, 16),
  'raw_qca_product_matches_target': False,
  'after_stabilizer_redefinition_matches_target': True}]